# Caso de estudio: RFM Parte 2 - Segmentación y Estrategias de Marketing

**Autor:** Robert Moreno

---

## Introducción

En este taller continuarás con el análisis RFM que realizaste en la Parte 1. Ahora que ya tienes calculadas las métricas de **Recency**, **Frequency** y **Monetary** para cada cliente, el siguiente paso es:

1. **Crear scores RFM** (del 1 al 5)
2. **Segmentar a los clientes** en grupos estratégicos
3. **Desarrollar estrategias de marketing** basadas en segmentos
4. **Exportar bases de clientes** para campañas específicas

---

## ⚠️ INSTRUCCIONES IMPORTANTES

A lo largo del notebook encontrarás código parcialmente completado. Debes buscar los siguientes marcadores:

- **`# TODO:`** - Indica una línea que DEBES completar
- **`# COMPLETAR:`** - Indica código que debes escribir
- **`# DESCOMENTAR:`** - Indica líneas que debes descomentar y posiblemente ajustar

**Ejemplo:**
```python
# TODO: Completa con el nombre correcto de la columna
df_filtrado = df[df[___] > 100]  # Reemplaza ___ con 'Monetary'
```

---

## Objetivos de aprendizaje

- Aplicar técnicas de scoring y segmentación
- Desarrollar estrategias de marketing data-driven
- Tomar decisiones basadas en análisis cuantitativo
- Comunicar insights de manera efectiva

## 1. Preparación de datos

**Instrucciones:**
1. Importa las librerías necesarias (ya están incluidas)
2. Carga tu tabla RFM que creaste en la Parte 1
3. Si no la guardaste, ejecuta el código para recrearla

In [1]:
# Librerías
import pandas as pd
import numpy as np
from datetime import datetime

print("✓ Librerías cargadas correctamente")

✓ Librerías cargadas correctamente


In [ ]:
# Carga de datos originales
# TODO: Completa las rutas de los archivos si están en otra ubicación
clients_df = pd.read_csv('../Tarea/Clients.csv')
ventas_df = pd.read_csv('../Tarea/ventas.csv')
facturas_df = pd.read_csv('../Tarea/facturas.csv')

print(f"✓ Clientes cargados: {len(clients_df)}")
print(f"✓ Ventas cargadas: {len(ventas_df)}")
print(f"✓ Facturas cargadas: {len(facturas_df)}")

✓ Clientes cargados: 60000
✓ Ventas cargadas: 1834174
✓ Facturas cargadas: 371000


In [5]:
# Recreación de la tabla RFM
# Este código recrea el análisis RFM de la Parte 1

# Convertir fechas
ventas_df['fecha_hora_compra'] = pd.to_datetime(ventas_df['fecha_hora_compra'])
facturas_df['fecha_hora_compra'] = pd.to_datetime(facturas_df['fecha_hora_compra'])

# Calcular monto de cada venta
# TODO: Completa la fórmula para calcular el monto (considera cantidad, precio_unitario y descuento_pct)
ventas_df['monto'] = ventas_df['cantidad'] * ventas_df['precio_unitario'] * (1 - ventas_df['descuento_pct'] / 100)

# Agrupar ventas por factura para obtener monto total
monto_por_factura = ventas_df.groupby('id_factura')['monto'].sum().reset_index()
monto_por_factura.columns = ['id_factura', 'monto_total']

# Unir con información de facturas
facturas_completas = facturas_df.merge(monto_por_factura, on='id_factura')

# Fecha de análisis (fecha más reciente en los datos)
fecha_analisis = facturas_completas['fecha_hora_compra'].max()
print(f"Fecha de análisis: {fecha_analisis.date()}")

# Calcular métricas RFM por cliente
rfm_df = facturas_completas.groupby('id_cliente').agg({
    'fecha_hora_compra': lambda x: (fecha_analisis - x.max()).days,  # Recency
    'id_factura': 'count',  # Frequency
    'monto_total': 'sum'  # Monetary
}).reset_index()

rfm_df.columns = ['id_cliente', 'Recency', 'Frequency', 'Monetary']

print(f"\n✓ Tabla RFM creada con {len(rfm_df)} clientes")
print("\nPrimeros registros:")
rfm_df.head()

Fecha de análisis: 2025-06-30

✓ Tabla RFM creada con 53000 clientes

Primeros registros:


,id_cliente,Recency,Frequency,Monetary
0,100000,43,3,176.4285
1,100002,29,16,1351.2545
2,100011,9,30,3674.8815
3,100016,0,9,899.6840
4,100037,56,2,84.0300


In [6]:
# Estadísticas descriptivas de RFM
print("Estadísticas de las métricas RFM:\n")
rfm_df[['Recency', 'Frequency', 'Monetary']].describe()

Estadísticas de las métricas RFM:



,Recency,Frequency,Monetary
count,53000.000000,53000.000000,53000.000000
mean,37.724396,7.000000,1255.662465
std,38.735579,9.954653,4908.206765
min,0.000000,1.000000,2.660000
25%,8.000000,2.000000,215.982625
50%,24.000000,4.000000,434.593250
75%,55.000000,7.000000,793.495625
max,180.000000,134.000000,90854.295000


---

## 2. Creación de Scores RFM

### ¿Qué son los scores RFM?

Los **scores RFM** son calificaciones numéricas (del 1 al 5) que se asignan a cada cliente:

- **Score de Recency (R_Score):** Score 5 = Compraron recientemente ✓
- **Score de Frequency (F_Score):** Score 5 = Compran frecuentemente ✓
- **Score de Monetary (M_Score):** Score 5 = Gastan mucho ✓

Usaremos el método de **quintiles** (división en 5 grupos iguales).

### 2.1 Calcular Score de Recency (R_Score)

In [ ]:
# Cálculo del R_Score
# IMPORTANTE: Para Recency, menor valor = mejor (compró más recientemente)
# Por eso invertimos los labels: [5,4,3,2,1]

# TODO: Descomentar y completar la siguiente línea
# rfm_df['R_Score'] = pd.qcut(rfm_df['___'], q=5, labels=[5,4,3,2,1])
# rfm_df['R_Score'] = rfm_df['R_Score'].astype(int)

# COMPLETAR: Descomenta la línea de arriba y reemplaza ___ con 'Recency'

# Verificación
# DESCOMENTAR la siguiente línea después de completar:
# print("Distribución de R_Score:\n", rfm_df['R_Score'].value_counts().sort_index())

### 2.2 Calcular Score de Frequency (F_Score)

In [ ]:
# Cálculo del F_Score
# Para Frequency, mayor valor = mejor
# Usamos duplicates='drop' por si hay muchos valores repetidos

# TODO: Completa la columna correcta y los labels correctos
#rfm_df['F_Score'] = pd.qcut(rfm_df[___], q=5, labels=[1,2,3,4,5], duplicates='drop')
#rfm_df['F_Score'] = rfm_df['F_Score'].astype(int)

print("Distribución de F_Score:\n", rfm_df['F_Score'].value_counts().sort_index())

### 2.3 Calcular Score de Monetary (M_Score)

In [ ]:
# Cálculo del M_Score
# TODO: Completa el código siguiendo el patrón de F_Score
# Recuerda: Mayor Monetary = mejor, entonces labels van de 1 a 5

# COMPLETAR: Descomenta y completa
# rfm_df['M_Score'] = pd.qcut(rfm_df[___], q=___, labels=[___], duplicates='drop')
# rfm_df['M_Score'] = rfm_df['M_Score'].astype(int)

# DESCOMENTAR después de completar:
# print("Distribución de M_Score:\n", rfm_df['M_Score'].value_counts().sort_index())

### 2.4 Crear RFM_Score combinado

In [ ]:
# Crear RFM_Score como concatenación de strings (ej: "555")
# TODO: Completa las columnas correctas
rfm_df['RFM_Score'] = (rfm_df['R_Score'].astype(str) + 
                        rfm_df['F_Score'].astype(str) + 
                        rfm_df['M_Score'].astype(str))

# Crear RFM_Score_Num como suma (para análisis numérico)
# TODO: Completa la fórmula sumando los tres scores
# rfm_df['RFM_Score_Num'] = rfm_df[___] + rfm_df[___] + rfm_df[___]

# DESCOMENTAR después de completar:
# print("\nTabla RFM con Scores:")
# rfm_df.head(10)

### 2.5 Análisis exploratorio de scores

In [ ]:
# Estadísticas de scores
print("Estadísticas de Scores:\n")
print(rfm_df[['R_Score', 'F_Score', 'M_Score', 'RFM_Score_Num']].describe())


---

## 3. Segmentación de Clientes

### ¿Qué es la segmentación RFM?

Agrupa a los clientes en categorías estratégicas:

| Segmento | Características | Estrategia |
|----------|----------------|------------|
| **Champions** | R=5, F=5, M=5 | Recompensas VIP |
| **Loyal Customers** | F alto, regulares | Upselling |
| **Big Spenders** | M alto | Productos premium |
| **Promising** | R alto, nuevos | Onboarding |
| **At Risk** | R bajo, antes buenos | Reactivación |
| **Hibernating** | R muy bajo | Reconquista |
| **Lost** | R=1, F=1, M=1 | Descuentos agresivos |

### 3.1 Función de segmentación

In [ ]:
def asignar_segmento(rfm_score):
    """
    Asigna un segmento basado en el RFM_Score (string de 3 dígitos)
    
    Parámetros:
    -----------
    rfm_score : str
        String de 3 dígitos representando R, F, M scores (ej: "555")
    
    Retorna:
    --------
    str : Nombre del segmento
    """
    # Extraer scores individuales
    r = int(rfm_score[0])
    f = int(rfm_score[1])
    m = int(rfm_score[2])
    
    # Champions: Los mejores en todo
    if r >= 5 and f >= 5 and m >= 5:
        return 'Champions'
    
    # Loyal Customers: Compran frecuentemente y recientemente
    elif r >= 4 and f >= 4:
        return 'Loyal Customers'
    
    # Big Spenders: Gastan mucho y están activos
    elif m >= 5 and r >= 3:
        return 'Big Spenders'
    
    # Promising: Nuevos con buen potencial (recientes pero baja frecuencia)
    elif r >= 5 and f <= 2:
        return 'Promising'
    
    # At Risk: Antes eran buenos (alta F/M), pero no han comprado recientemente
    # TODO: Completa la condición
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'
    
    # Hibernating: Inactivos pero antes compraban algo
    elif r <= 2 and f <= 2 and m >= 2:
        return 'Hibernating'
    
    # Lost: Perdidos completamente
    # TODO: Completa la condición para clientes con todos los scores bajos
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost'
    
    # Need Attention: Requieren atención antes de perderse
    elif r <= 3 and f <= 3:
        return 'Need Attention'
    
    # About to Sleep: Están por volverse inactivos
    elif r <= 3 and f >= 2:
        return 'About to Sleep'
    
    # Others: No caen en ninguna categoría específica
    else:
        return 'Others'

print("✓ Función de segmentación definida")

### 3.2 Aplicar segmentación

In [ ]:
# Aplicar la función de segmentación a cada cliente
# TODO: Descomenta y completa aplicando la función a la columna RFM_Score
#rfm_df['Segmento'] = rfm_df['RFM_Score'].apply(___)

# Distribución de clientes por segmento
print("Distribución de clientes por segmento:\n")
segmento_counts = rfm_df['Segmento'].value_counts()
print(segmento_counts)
print(f"\nTotal de clientes: {len(rfm_df)}")

### 3.3 Incluyendo Ticket Promedio

In [ ]:
# TODO: Calcule el ticket promedio de cada cliente y guardelo en un dataframe llamado df_tp
# Recordar que ticket promedio es el valor $$ que el cliente gasta en promedio en cada compra. Puedes utilizar la doble agrupación, 
# O calcular para cada cliente monto total / numero de facturas 



In [ ]:
# TODO:  realice el merge entre rfm_df con df_tp

## 4. Elije UNO DE ESTOS DOS CASOS

---

### 4.1 Caso 1: Programa de Lealtad "Club Black"

#### 🌟 Contexto del negocio

La empresa está lanzando un programa de lealtad exclusivo llamado **"Club Black"**, diseñado para reconocer y recompensar a los mejores clientes.

**Beneficios del programa:**
- Descuentos exclusivos del 15-20%
- Acceso anticipado a nuevos productos
- Asesoría personalizada
- Eventos VIP exclusivos
- Envío gratuito en todas las compras
- Línea de atención prioritaria

#### 🎯 Tu tarea:

1. **Seleccionar los clientes** que formarán parte del Club Black basándote en la segmentación RFM
2. **Exportar la base de datos** de estos clientes en formato CSV (`club_black_clientes.csv`)
3. **Realizar un análisis que permita describir al grupo seleccionado**: 
   (Por ejemplo)
   - ¿Que porcentaje representan sobre el total de clientes?
   - ¿Cuál es su ticket promedio comparado con el resto de clientes?
   - ¿Cuál es su número de compras (frecuencia) comparado con el resto?
   - ¿Qué porcentaje de los ingresos totales representan?
   - Cualquier otro insight relevante que justifique su inclusión en el programa

#### 💡 Preguntas guía:

- ¿Qué segmentos RFM incluirías en el Club Black?
- ¿Deberías agregar algún filtro adicional? (ej: Monetary mínimo, Recency máximo)
- ¿Cómo se comparan estos clientes con el resto de la base?
- ¿Qué valor aportan a la empresa?

#### 📝 Instrucciones:

En las siguientes celdas, **escribe tu propio código** para:
1. Filtrar y seleccionar los clientes del Club Black
2. Realizar los análisis comparativos solicitados
3. Exportar la base con `.to_csv()`

---

### 4.2. Caso 2: Campaña de Recuperación de Clientes

#### 💰 Contexto del negocio

La empresa ha notado una disminución en la actividad de varios clientes que antes eran valiosos. Se ha decidido lanzar una **Campaña de Recuperación** con los siguientes parámetros:

**Presupuesto de la campaña:**
- **5,000 clientes** pueden ser contactados (presupuesto máximo)
- **Costo:** $5 USD por cliente contactado
- **Presupuesto total:** $50,000 USD
- **Estrategia:** Email marketing personalizado + cupones de descuento
**Objetivo:** Reactivar clientes que se están alejando o ya están inactivos, maximizando la probabilidad de recuperación .

#### 🎯 Tu tarea:

1. **Seleccionar estratégicamente** hasta 5,000 clientes que sean candidatos para recuperación
2. Si hay más de 5,000 candidatos, **priorizar** y argumentar el criterio
3. **Exportar la base de datos** de estos clientes en formato CSV (`recuperacion_clientes.csv`)
4. **Justificar tu estrategia** de selección y priorización

#### 💡 Preguntas guía:

- ¿Qué segmentos RFM son candidatos para recuperación? 
- Si tienes más de 5,000 candidatos, ¿cómo los priorizarías?


#### 📝 Instrucciones:

En las siguientes celdas, **escribe tu propio código** para:
1. Identificar candidatos para recuperación
2. Priorizarlos si es necesario (máximo 5,000)
3. Realizar análisis del perfil de estos clientes
4. Exportar la base con `.to_csv()`

In [7]:
## TODO: UNA VEZ ELIJA EL CASO, DEBE REALIZAR EL RESPECTIVO ANÁLISIS, AQUÍ
## COMENTE SUS DECISIONES

## 5. BONUS: Realice una PPT/One pager (SOLO 1 SLIDE)
## Debe ser presentando 1 de estas 3 estrategias: 
* Se desea presentar en 1 sola slide los resultados de la segmentación RFM, donde se detalle cada grupos, características, posibles estrategias a desarrollar para algunos de estos segmentos. (Debe contener datos, insigths al menos de grupos más importantes. Por ejemplo: el ticket promedio que maneja cada grupo, frecuencia, promedio, cuanto % de las ventas representan)
* Se necesita resumir la estrategia de lealtad del programa Club Black, realice una slide que pueda ser presentada a gerencia con el resumen y características de los clientes seleccionados. 
* Se necesita resumir la estrategia de recuperacuón de clientes, realice una slide que pueda ser presentada a gerencia con el resumen y características de los clientes seleccionados. 

NOTA: PARA LA REALIZACIÓN DE ESTA SLIDE, ESTÁ COMPLETAMENTE PERMITIDO EL USO DE HERRAMIENTAS DE IA GENERATIVA COMO CLAUDE O GEMINI, 
recomendacion: Estructuren bien el prompt incluyendo los datos (insights) relevantes que quieren visualizar.  

##### Todo el que haga el bonus y esté bien hecho(OJO):
Premios: 
* +1 puntos extra en cualquier actividad/tarea/taller

**3 mejores One pager (Revisadas en clase antes de las 12pm):**  
Premios:
* Safisfacción de haber ganado.
* Mini-tictac
* +2 puntos extra en cualquier actividad/tarea/taller

---

## 🎉 ¡Excelente trabajo!

Has completado un análisis completo de segmentación RFM y desarrollado estrategias de marketing data-driven. 

### 🎓 Habilidades desarrolladas:

- ✅ Scoring y segmentación de clientes con metodología RFM
- ✅ Análisis comparativo de grupos de clientes
- ✅ Toma de decisiones estratégicas basada en datos
- ✅ Priorización de recursos en campañas de marketing
- ✅ Uso de IA generativa para comunicación ejecutiva
- ✅ Exportación y preparación de datos para campañas
- ✅ Análisis descriptivo y generación de insights de negocio

### 💼 Aplicaciones profesionales:

Estas habilidades son fundamentales en roles de:
- **Marketing Analytics**
- **Customer Intelligence**
- **CRM Strategy**
- **Business Analytics**
- **Data Science aplicado a negocio**
- **Customer Success Management**

### 🚀 Próximos pasos:

1. **Machine Learning:** Predecir qué clientes están en riesgo de churn antes de que suceda
2. **CLV (Customer Lifetime Value):** Predecir el valor futuro de cada cliente
3. **Clustering avanzado:** K-means, DBSCAN para descubrir segmentos no obvios
4. **A/B Testing:** Probar diferentes mensajes y ofertas por segmento

---

### 📧 Entrega final:

**Asegúrate de enviar:**

1. ✅ Este notebook completado (`.ipynb`)
2. ✅ Los 3 archivos CSV generados:
   - `resumen_segmentos.csv`
   - `club_black_clientes.csv`
   - `recuperacion_clientes.csv`
3. ✅ La presentación generada con IA:
   - `presentacion_club_black.pdf` (o `.png`)
4. ✅ (Opcional) Un archivo ZIP con todo lo anterior

---

**¡Éxito en tu carrera como Data Analyst! 📊🚀**

Recuerda que la segmentación de clientes es una de las aplicaciones más valiosas del análisis de datos en el mundo real. Las habilidades que desarrollaste aquí son directamente aplicables a cualquier industria que tenga clientes.